In [ ]:
# =========================================================
# XGBOOST + SARIMAX + SEASONAL NAIVE FORECAST COMPARISON
# =========================================================

import sys
sys.path.append("../src")

import pandas as pd

from xgboost_utils import (
    get_xgboost_feature_sets,
    fit_predict_xgboost_models,
    seasonal_naive_forecast,
    build_sarimax_forecasts,
    build_comparison_plot_df,
    get_xgb_gain_importance,
    get_xgb_shap_importance,
    make_pretty_feature_names,
    build_shap_comparison_table,
    save_shap_outputs
)

In [ ]:
# =========================================================
# 1. XGBOOST FEATURE SETS
# =========================================================

feature_sets = get_xgboost_feature_sets(
    df=df,
    development_data=development_data,
    country=country,
    lag_prefix=lag_prefix,
    full_exog_vars=full_exog_vars,
    restricted_exog_vars=restricted_exog_vars
)

for model_name, features in feature_sets.items():
    print("=" * 70)
    print(model_name)
    print(features)

In [ ]:
# =========================================================
# 2. FIT AND FORECAST XGBOOST MODELS
# =========================================================

xgb_predictions_log, fitted_xgb_models, xgb_test_targets = fit_predict_xgboost_models(
    development_data=development_data,
    forecast_data=forecast_data,
    target_col=target_col,
    feature_sets=feature_sets
)

In [ ]:
# =========================================================
# 3. BUILD SARIMAX FORECASTS
# =========================================================

sarimax_predictions_log = build_sarimax_forecasts(
    forecast_data=forecast_data,
    target_col=target_col,
    model_specs=model_specs,
    fitted_sarimax_models=fitted_sarimax_models
)

In [ ]:
# =========================================================
# 4. SEASONAL NAIVE FORECAST
# =========================================================

naive_pred_log = seasonal_naive_forecast(
    df=df,
    target_col=target_col,
    forecast_data=forecast_data,
    seasonal_lag=7
)

In [ ]:
# =========================================================
# 5. COMBINE ALL FORECASTS
# =========================================================

predictions_log = {}

predictions_log.update(sarimax_predictions_log)
predictions_log.update(xgb_predictions_log)

predictions_log["Seasonal Naïve"] = naive_pred_log

y_test_log = forecast_data[target_col].copy()

comparison_plot_df = build_comparison_plot_df(
    y_test_log=y_test_log,
    predictions_log=predictions_log
)

print(comparison_plot_df.shape)
comparison_plot_df.head()

XGBOOST FEATURE IMPORTANCE

In [ ]:
# Recreate train/test matrices for SHAP
xgb_data = {}

for model_name, features in feature_sets.items():
    train = development_data[[target_col] + features].dropna().copy()
    test = forecast_data[[target_col] + features].dropna().copy()

    xgb_data[model_name] = {
        "X_train": train[features],
        "X_test": test[features]
    }

In [ ]:
gain_importance_dict = {}
shap_values_dict = {}
shap_importance_dict = {}

for model_name, model in fitted_xgb_models.items():
    X_train = xgb_data[model_name]["X_train"]
    X_test = xgb_data[model_name]["X_test"]

    gain_importance_dict[model_name] = get_xgb_gain_importance(
        model=model,
        feature_names=X_train.columns
    )

    shap_values, shap_importance = get_xgb_shap_importance(
        model=model,
        X_test=X_test
    )

    shap_values_dict[model_name] = shap_values
    shap_importance_dict[model_name] = shap_importance

    print(f"\n{model_name} - Gain importance")
    print(gain_importance_dict[model_name].head(15))

    print(f"\n{model_name} - SHAP importance")
    print(shap_importance_dict[model_name].head(15))

In [ ]:
pretty_feature_names = make_pretty_feature_names(
    country=country,
    code=code,
    lag_prefix=lag_prefix
)

shap_compare = build_shap_comparison_table(
    shap_importance_dict=shap_importance_dict,
    pretty_feature_names=pretty_feature_names
)

shap_compare.head(15)

In [ ]:
excel_path, csv_path = save_shap_outputs(
    country=country,
    forecast_year=forecast_year,
    gain_importance_dict=gain_importance_dict,
    shap_importance_dict=shap_importance_dict,
    shap_compare=shap_compare,
    output_dir="outputs/shap"
)

print(f"Saved Excel: {excel_path}")
print(f"Saved CSV: {csv_path}")